[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/example2_mixed_feedback.ipynb)

# Example 2 - When a good fit lands in the *wrong* region

This is the example that motivates the whole paper, so it is worth slowing down on.

In Example 1 the toggle switch was forgiving: every recovery kept the right qualitative
behaviour. Here we meet a circuit where that breaks. A least-squares fit can look perfectly
reasonable on the data and yet, once the data are noisy, place the model in a **different**
DSGRN region - a different qualitative regime. That is the gap between *fitting numbers* and
*recovering dynamics*, made concrete.

Quick recap of the idea (each notebook stands alone). **DSGRN** partitions parameter space
into finitely many regions; within one region the qualitative dynamics is fixed. A trial
achieves **region recovery** when the learned parameters satisfy the *same* defining
inequalities as the truth - a single all-or-nothing check, no DSGRN install needed.

The circuit is a two-gene **mixed-feedback** network ($\gamma=1$): gene `x1` self-activates
and is activated by `x2` (their effects **add**, an OR-type rule), while gene `x2` is
repressed by `x1` and self-activates (their effects **multiply**). We sample parameters
from DSGRN parameter node **712**.

In [ ]:
# Colab already has numpy/scipy/matplotlib/torch, so this cell is a no-op there.
# Uncomment if you are running somewhere that is missing a package.
# !pip -q install numpy scipy matplotlib torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import torch, torch.nn as nn
torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=3, suppress=True)

### The smooth switches

As in Example 1, production is built from Hill functions - smooth switches between a low
level `L` and a high level `U` at a threshold `theta`, with sharpness `d`. `f+` activates
(low below `theta`, high above); `f-` represses (the reverse). Here some inputs add and one
pair multiplies, which is what makes the dynamics richer than a plain toggle.

In [ ]:
# --- Hill production functions: smooth switches (gamma normalized to 1) ---
def hill_act(x, L, U, th, d):   # activating: low L -> high U as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * xd / (td + xd)

def hill_rep(x, L, U, th, d):   # repressing: high U -> low L as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * td / (td + xd)

In [ ]:
# --- the same two functions in torch (used inside the network's physics loss) ---
def hill_act_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * xd / (td + xd)

def hill_rep_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * td / (td + xd)

## 1. The model and its DSGRN region (node 712)

$$\dot{x}_1=-x_1+\underbrace{f^+(x_1)+f^+(x_2)}_{\text{additive}},\qquad
\dot{x}_2=-x_2+\underbrace{f^-(x_1)\,f^+(x_2)}_{\text{multiplicative}}.$$

The region for node 712 is a chain of inequalities ordering the thresholds against sums of
production levels (for the additive gene `x1`) and against products of production levels
(for the multiplicative gene `x2`). You do not need to memorize it - `in_region` below is
that chain, written out. Conceptually it is the same move as the toggle's
`0 < L < gamma*theta < U`, just with more terms because the circuit is bigger.

In [ ]:
P = dict(L11=1.0,U11=4.0, L21=1.0,U21=2.0, th11=4.0, th12=2.5,
         L12=1.0,U12=2.0, L22=1.0,U22=4.0, th21=1.5, th22=3.0, d=10.0, g=1.0)

def rhs(t, x, p):
    x1, x2 = x
    prod1 = (hill_act(x1, p['L11'],p['U11'],p['th11'],p['d'])
             + hill_act(x2, p['L21'],p['U21'],p['th21'],p['d']))
    prod2 = (hill_rep(x1, p['L12'],p['U12'],p['th12'],p['d'])
             * hill_act(x2, p['L22'],p['U22'],p['th22'],p['d']))
    return [-p['g']*x1 + prod1, -p['g']*x2 + prod2]

def in_region(p):   # DSGRN node 712: orderings of thresholds vs production sums/products
    return (
        p['L11']+p['L21'] < p['th12'] < p['L11']+p['U21'] < p['th11']
            < p['U11']+p['L21'] < p['U11']+p['U21'] and
        p['L12']*p['L22'] < p['th21'] < p['U12']*p['L22'] < p['th22']
            < p['L12']*p['U22'] < p['U12']*p['U22'] and
        0 < p['L11'] < p['U11'] and 0 < p['L21'] < p['U21'] and
        0 < p['th12'] < p['th11'] and 0 < p['L12'] < p['U12'] and
        0 < p['L22'] < p['U22'] and 0 < p['th21'] < p['th22'])

print('do the true parameters lie in node 712?', in_region(P))

In [ ]:
def simulate(rhs, p, x0, T, n):
    t = np.linspace(0.0, T, n)
    sol = solve_ivp(rhs, (0.0, T), x0, t_eval=t, args=(p,), rtol=1e-9, atol=1e-11)
    return t, sol.y.T   # times (n,), states (n, m)

def add_noise(y, ub_frac, scale, rng):
    yn = y + rng.uniform(-ub_frac*scale, ub_frac*scale, size=y.shape)
    return np.clip(yn, 0.0, None)   # concentrations stay non-negative

## 2. Simulate the circuit and add noise

Same recipe as before: simulate the true system from several initial conditions to make
synthetic data, then corrupt it with bounded noise. With more parameters and a
multiplicative term, this circuit is harder to pin down from data - which is exactly why it
is a good stress test.

In [ ]:
T, n = 20.0, 80
x0s = [[0.5,0.5],[3.0,1.0],[1.0,4.0],[5.0,5.0],[2.0,3.0]]
ts, xs = [], []
for x0 in x0s:
    t, y = simulate(rhs, P, x0, T, n); ts.append(t); xs.append(y)

gap = min(abs(P['L11']-P['U11']), abs(P['L21']-P['U21']),
          abs(P['L12']-P['U12']), abs(P['L22']-P['U22']))
rng = np.random.default_rng(0)
xs_noisy = [add_noise(y, 0.25, gap, rng) for y in xs]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for t, y, yn in zip(ts, xs, xs_noisy):
    ax[0].plot(t, yn[:,0], '.', ms=3); ax[0].plot(t, y[:,0], 'k-', lw=.6, alpha=.4)
    ax[1].plot(t, yn[:,1], '.', ms=3); ax[1].plot(t, y[:,1], 'k-', lw=.6, alpha=.4)
ax[0].set_title('$x_1$ (noisy)'); ax[1].set_title('$x_2$ (noisy)')
plt.tight_layout(); plt.show()

## 3. Least squares, and the moment it goes wrong

We run the same gradient-matching least squares as in Example 1, now over all twelve
parameters. Then comes the punchline of the notebook: we sweep the noise level and watch
the region-recovery rate. Keep your eye on how it changes - this is the figure the paper is
built around.

In [ ]:
KEYS = ['L11','U11','L21','U21','th11','th12','L12','U12','L22','U22','th21','th22']
def ls_recover(ts, xs, d=10.0, g=1.0):
    X = np.vstack(xs)
    DX = np.vstack([np.gradient(y, t, axis=0) for t, y in zip(ts, xs)])
    def resid(z):
        p = dict(zip(KEYS, z))
        f1 = (-g*X[:,0] + hill_act(X[:,0],p['L11'],p['U11'],p['th11'],d)
              + hill_act(X[:,1],p['L21'],p['U21'],p['th21'],d))
        f2 = (-g*X[:,1] + hill_rep(X[:,0],p['L12'],p['U12'],p['th12'],d)
              * hill_act(X[:,1],p['L22'],p['U22'],p['th22'],d))
        return np.concatenate([DX[:,0]-f1, DX[:,1]-f2])
    z0 = np.array([1.5,3.,1.5,1.8,3.5,2.2,1.2,1.8,1.2,3.5,1.3,2.5])
    s = least_squares(resid, z0, bounds=(0, 30), max_nfev=20000)
    p = dict(zip(KEYS, s.x)); p['d'] = d; p['g'] = g; return p

print('LS at 25% noise lands in node 712:', in_region(ls_recover(ts, xs_noisy)))

levels = [0.0, 0.1, 0.25, 0.5]; n_trials = 20; rate = []
for ub in levels:
    r = np.random.default_rng(7); hits = 0
    for _ in range(n_trials):
        xs_n = [add_noise(y, ub, gap, r) for y in xs]
        hits += in_region(ls_recover(ts, xs_n))
    rate.append(hits / n_trials)
plt.plot([100*l for l in levels], rate, 'o-')
plt.xlabel('noise upper bound (%)'); plt.ylabel('region-recovery rate (LS)')
plt.ylim(-0.05, 1.05); plt.title('Mixed feedback: least squares drifts out of the region')
plt.show()
print('LS region-recovery rate by noise:', dict(zip(levels, rate)))

**What you should see.** With clean data least squares recovers the region almost every
time. As the noise grows the rate **falls off a cliff** - by 50% noise it is near zero. The
fits still look like plausible trajectories, but the *parameters* have crossed a region
boundary, so the model now describes the wrong qualitative regime. Parameter error alone
would not have told you this happened; the region check does.

In [ ]:
def build_tensors(ts, xs, x0s):
    # flatten all trajectories into (N,1) time, (N,m) repeated x0, (N,m) measured states
    t_d = np.concatenate([t[:, None] for t in ts])
    x_d = np.vstack(xs)
    x0_d = np.vstack([np.tile(x0, (len(t), 1)) for x0, t in zip(x0s, ts)])
    t_ic = np.zeros((len(x0s), 1)); x_ic = np.array(x0s, dtype=float)  # t=0 anchors
    to = lambda a: torch.tensor(a)
    return (to(t_d), to(x0_d), to(x_d), to(t_ic), to(x_ic))

## 4. Does the physics constraint help? A PINN demonstration

The paper's finding is that enforcing the differential equation during learning keeps the
PINN in the correct region where least squares wanders off. Below is a single PINN run
(twelve learnable parameters, neutral init) so you can see it end to end. A full
rate-vs-noise comparison would train many PINNs and is slow on CPU, so we keep it to one
demonstration; on a GPU you can raise `steps` and loop over noise levels yourself.

In [ ]:
# --- the physics-informed network: surrogate u_theta(t, x0) + learnable (L,U,theta,d) ---
class PINN(nn.Module):
    def __init__(self, m, param_init, T, hidden=64, depth=4, fourier_k=0):
        super().__init__()
        self.m, self.T, self.fourier_k = m, T, fourier_k
        in_dim = (2*fourier_k if fourier_k > 0 else 1) + m
        if fourier_k > 0:
            self.register_buffer('freqs', 2*np.pi*torch.arange(1, fourier_k+1).double())
        layers, d0 = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d0, hidden), nn.Tanh()]; d0 = hidden
        layers += [nn.Linear(d0, m)]
        self.net = nn.Sequential(*layers)
        # physical parameters via positive reparametrization  p = raw^2 + eps
        self.raw = nn.ParameterDict(
            {k: nn.Parameter(torch.tensor(float(v)**0.5)) for k, v in param_init.items()})

    def phys_params(self):
        return {k: self.raw[k]**2 + 1e-6 for k in self.raw}

    def _feat(self, t):
        tn = t / self.T   # normalize time to [0,1]; the chain rule is handled by autograd
        if self.fourier_k > 0:
            return torch.cat([torch.sin(self.freqs*tn), torch.cos(self.freqs*tn)], dim=1)
        return tn

    def forward(self, t, x0):
        return self.net(torch.cat([self._feat(t), x0], dim=1))

def fit_pinn(model, data, rhs_t, steps=4000, lr=1e-3, w=(5.0, 1.0, 1.0), log=500):
    t_d, x0_d, x_d, t_ic, x_ic = data   # all tensors
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for it in range(steps):
        opt.zero_grad()
        loss_data = ((model(t_d, x0_d) - x_d)**2).mean()           # fit the measurements
        tc = t_d.clone().requires_grad_(True)                       # physics residual:
        u = model(tc, x0_d)                                         #   du/dt should equal f(u)
        grads = [torch.autograd.grad(u[:, j].sum(), tc, create_graph=True)[0]
                 for j in range(model.m)]
        du = torch.cat(grads, dim=1)
        loss_phys = ((du - rhs_t(u, model.phys_params()))**2).mean()
        loss_ic = ((model(t_ic, x_ic) - x_ic)**2).mean()           # match initial conditions
        loss = w[0]*loss_data + w[1]*loss_phys + w[2]*loss_ic
        loss.backward(); opt.step()
        if log and it % log == 0:
            print(f'  step {it:5d}  data {loss_data.item():.2e}  phys {loss_phys.item():.2e}')
    return model

In [ ]:
def rhs_t(x, p):
    x1, x2 = x[:,0:1], x[:,1:2]
    prod1 = (hill_act_t(x1,p['L11'],p['U11'],p['th11'],p['d'])
             + hill_act_t(x2,p['L21'],p['U21'],p['th21'],p['d']))
    prod2 = (hill_rep_t(x1,p['L12'],p['U12'],p['th12'],p['d'])
             * hill_act_t(x2,p['L22'],p['U22'],p['th22'],p['d']))
    return torch.cat([-p['g']*x1 + prod1, -p['g']*x2 + prod2], dim=1)

init = {k: (0.05 if k[0]=='L' else 1.0) for k in KEYS}   # neutral: L~0, U~1, theta~1
init.update(d=5.0, g=1.0)
torch.manual_seed(0)
model = PINN(m=2, param_init=init, T=T, hidden=64, depth=4)
data = build_tensors(ts, xs_noisy, x0s)
model = fit_pinn(model, data, rhs_t, steps=6000, lr=1e-3, w=(5.0,1.0,1.0))

pp = {k: float(v) for k, v in model.phys_params().items()}
print('PINN estimate:', {k: round(pp[k],2) for k in KEYS})
print('PINN lands in node 712:', in_region(pp))

---
### The takeaway

The mixed-feedback circuit is the sharp case. A numerically reasonable least-squares fit can
move the model into a neighbouring DSGRN region under noise - a genuinely different
qualitative regime - and **region recovery flags exactly that failure** while a small
trajectory error would not. This is the complementarity at the heart of the paper: a close
fit is not the same as the right dynamics.

## Morse graph + Conley-index recovery (DSGRN, optional)

The region test above asks for **exact DSGRN region equality**. A weaker, biologically
natural criterion asks only that the recovered region carry the **same Conley-Morse graph**
as the target - the same recurrent Morse sets, reachability order, and Conley-index labels -
up to label-preserving isomorphism. The implication

$$\text{same region}\ \Rightarrow\ \text{same Morse graph and Conley labels}$$

always holds, so region recovery already *implies* Morse/Conley recovery; the question is
whether the **converse** adds any successes - i.e. whether a miss on the exact region can
still land in a *different* region with an isomorphic Conley-Morse graph.

We use the existing DSGRN tools, not a new pipeline:
`par_index_from_sample` (learned parameters -> region index), `DSGRN.Blowup.ConleyMorseGraph`
(region -> annotated Morse graph), and `DSGRN.isomorphic_morse_graphs` (directed-graph
isomorphism with `node_match` on the **Conley index** - which is location-free, so two fixed
points in different cells match, while a fixed point never matches a periodic orbit).

In [ ]:
# Optional: building DSGRN needs a C++ toolchain (a few minutes on Colab).
# !pip -q install networkx DSGRN
DSGRN_OK = False
try:
    import DSGRN, numpy as np
    NET_SPEC = 'x : x+y\ny : (~x)y\n'
    _net = DSGRN.Network(NET_SPEC)
    _pg  = DSGRN.ParameterGraph(_net)
    TARGET = 712
    _mg = {}
    def conley_morse(idx):          # region index -> annotated Conley-Morse graph (cached)
        if idx not in _mg:
            _mg[idx] = DSGRN.Blowup.ConleyMorseGraph(_pg.parameter(idx))[0]
        return _mg[idx]
    def to_matrices(p):             # learned (L, U, theta) -> DSGRN L, U, T matrices
        n = 2
        L = np.zeros((n, n)); U = np.zeros((n, n)); T = np.zeros((n, n))
        L[0,0],U[0,0]=p['L11'],p['U11']; L[1,0],U[1,0]=p['L21'],p['U21']
        L[0,1],U[0,1]=p['L12'],p['U12']; L[1,1],U[1,1]=p['L22'],p['U22']
        T[0,0],T[0,1],T[1,0],T[1,1]=p['th11'],p['th12'],p['th21'],p['th22']
        return L, U, T
    def region_of(p):               # learned parameters -> DSGRN region index (-1 if outside)
        L, U, T = to_matrices(p)
        return DSGRN.par_index_from_sample(_pg, L, U, T)
    def morse_recovers(idx, target=TARGET):   # same Conley-Morse graph up to label-preserving iso
        return idx >= 0 and DSGRN.isomorphic_morse_graphs(conley_morse(idx), conley_morse(target))
    print('target region', TARGET, 'Conley indices:',
          [conley_morse(TARGET).vertex_label(v)[2] for v in conley_morse(TARGET).vertices()])
    DSGRN_OK = True
except Exception as e:
    print('DSGRN unavailable - skipping this section:', repr(e)[:90])

In [ ]:
if DSGRN_OK:
    # the recovered parameters from the cells above
    p_ls = ls_recover(ts, xs_noisy)
    print('recovered region index:', region_of(p_ls),
          '| exact region match:', region_of(p_ls) == TARGET,
          '| Morse/Conley match:', morse_recovers(region_of(p_ls)))

    # noise sweep: exact-region recovery vs Morse/Conley recovery (least squares)
    levels = [0.0, 0.1, 0.25, 0.5]
    print(f'{"noise":>6} | {"exact region":>12} | {"Morse/Conley":>12} | {"region-miss, Morse-ok":>20}')
    for ub in levels:
        r = np.random.default_rng(7); reg = mor = gapc = 0
        for _ in range(15):
            xs_n = [add_noise(y, ub, gap, r) for y in xs]
            idx = region_of(ls_recover(ts, xs_n))
            er = (idx == TARGET); mr = morse_recovers(idx)
            reg += er; mor += mr; gapc += (mr and not er)
        print(f'{ub:>6} | {reg:>10}/15 | {mor:>10}/15 | {gapc:>18}/15')
    # mixed feedback: node 712 shares its Conley-Morse graph with 3 other regions

**What you should see.** Here the criteria can disagree. DSGRN node 712 is *not* the only
region with its Conley-Morse graph - three other regions ({307, 987, 1392}) carry an
isomorphic one. So at moderate noise a few trials miss the exact region yet land in one of
those, and the `region-miss, Morse-ok` column is small but nonzero (about `1/15` near 25-50%
noise). Morse/Conley recovery is therefore a *strictly looser* criterion for this network -
the only one of the three where the choice of criterion changes the count.